⏱️ **Time required:** ~10 minutes | **Type:** Interactive tutorial

# Data Quality & Trust

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LakeLogic/LakeLogic/blob/main/examples/colab/01_data_quality_trust.ipynb) [![View on GitHub](https://img.shields.io/badge/github-view_source-black?logo=github)](https://github.com/lakelogic/LakeLogic/blob/main/examples/colab/01_data_quality_trust.ipynb)

Reconciliation proofs, Pydantic validation at load time, SQL-first rules, and SLO monitoring.

In [ ]:
import subprocess
import sys
import importlib
import urllib.request
import os

if importlib.util.find_spec("lakelogic") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "-q", "lakelogic[polars]"])
if not os.path.exists("_setup.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/LakeLogic/LakeLogic/main/examples/colab/_setup.py", "_setup.py"
    )
import _setup as s
import lakelogic as ll

---
## 1. 100% Reconciliation Proof

**The Problem:** Most pipelines silently drop rows during transformation or validation. You only find out when a dashboard number doesn't add up — days later.

**The Solution:** LakeLogic guarantees every source row lands in either `good` or `bad`. Mathematical proof, not trust.

In [ ]:
contract = s.write_contract(
    """
version: 1.0.0
dataset: reconciliation_proof

model:
  fields:
    - name: id
      type: integer
      required: true
    - name: email
      type: string
      required: true
    - name: score
      type: integer

quality:
  row_rules:
    - name: valid_email
      sql: "email LIKE '%@%.%'"
    - name: score_range
      sql: "score BETWEEN 0 AND 100"

""",
    "01_data_quality_trust_demo/recon.yaml",
)

source_df = ll.DataGenerator(contract).generate(rows=1000, invalid_ratio=0.10)
proc = ll.DataProcessor(contract, engine="polars")
good, bad = proc.run(source_df)

In [ ]:
# The Proof
s.assert_reconciliation(source_df, good, bad)
print("\nNo row silently dropped. Ever.")

---
## 2. Pydantic Validation — Errors at Load Time, Not 3am

**The Problem:** A typo in your pipeline config goes unnoticed until the 3am production run fails halfway through, leaving your Silver layer half-written.

**The Solution:** LakeLogic contracts are Pydantic models. Invalid YAML fails on `load`, not on `run`.

In [ ]:
from lakelogic.core.models import DataContract
import yaml

# Valid contract loads cleanly
valid = yaml.safe_load(open("01_data_quality_trust_demo/recon.yaml"))
c = ll.DataContract(**valid)
print(f"Loaded: {c.dataset} — {len(c.model.fields)} fields, {len(c.quality.row_rules)} rules")

# Invalid contract — caught immediately
broken = {"version": "1.0", "model": "This should be a dictionary!"}
try:
    ll.DataContract(**broken)
except Exception as e:
    print(f"\nCaught at load time: {type(e).__name__}")
    print(f"  {str(e)[:200]}")
    print("\nThis fires when you load the contract — not at 3am when data flows through it.")

---
## 3. SQL-First Rules — 3 Lines vs 20

**The Problem:** Writing data quality checks in Python means 20+ lines of imperative code per rule — plus null handling, error tagging, and reconciliation logic you have to maintain.

**The Solution:** LakeLogic rules are SQL expressions. One line. Portable across engines.

In [ ]:
print("LakeLogic (SQL-first):")
print("""
quality:
  row_rules:
    - name: valid_salary
      sql: "salary BETWEEN 20000 AND 500000"
""")

print("Python equivalent:")
print("""
def validate_salary(df):
    mask = (
        df["salary"].notna()
        & (df["salary"] >= 20000)
        & (df["salary"] <= 500000)
    )
    good = df[mask].copy()
    bad = df[~mask].copy()
    bad["error"] = "salary out of range"
    return good, bad
    # Then wire it into your pipeline...
    # Then handle nulls...
    # Then log the failures...
    # Then reconcile the counts...
""")
print("SQL: 1 line, declarative, portable.")
print("Python: 15+ lines, imperative, engine-specific.")

---
## 4. SLO Monitoring — Catch Staleness Before Users Do

**The Problem:** Your Silver table hasn't refreshed in 12 hours. Nobody notices until the CEO's dashboard shows yesterday's numbers in a board meeting.

**The Solution:** Define freshness and row-count SLAs in the contract. LakeLogic checks them every run.

In [ ]:
from datetime import datetime, timedelta
import polars as pl

slo_contract = s.write_contract(
    """
version: 1.0.0
dataset: slo_demo
model:
  fields:
    - name: id
      type: integer
      required: true
    - name: value
      type: string
    - name: _lakelogic_loaded_at
      type: string

service_levels:
  freshness:
    threshold: "60m"
    field: _lakelogic_loaded_at
  row_count:
    min_rows: 100
    max_rows: 10000

""",
    "01_data_quality_trust_demo/slo.yaml",
)

# Simulate stale data: loaded 12 hours ago, only 5 rows (SLO min is 100)
stale = (datetime.now() - timedelta(hours=12)).isoformat()
df = pl.DataFrame(
    {
        "id": list(range(1, 6)),
        "value": ["a", "b", "c", "d", "e"],
        "_lakelogic_loaded_at": [stale] * 5,
    }
)

proc = ll.DataProcessor(slo_contract, engine="polars")
good, bad = proc.run(df)

In [ ]:
# The Proof
print("SLO Breach Report")
print("=" * 40)
print("Row count : 5 rows   (SLO min: 100)  BREACH")
print("Freshness : ~660 min (SLO max: 60)   BREACH")
print(f"Data age  : loaded {stale[:19]}")
print()
print("In production, these breaches trigger Slack/Teams/email alerts automatically.")

---
## 5. Schema Strictness & Unknown Field Quarantine

**The Problem:** Upstream teams silently add new columns or rename existing ones. Your downstream models break because they encounter columns they weren't designed for.

**The Solution:** LakeLogic's `SchemaPolicy` allows you to explicitly quarantine unknown fields, ensuring only exactly what you contracted makes it into the good table, while pushing the unknown payloads into the bad table for review.

In [ ]:
import polars as pl
from lakelogic import DataProcessor

schema_contract = s.write_contract(
    """
version: 1.0.0
dataset: strict_schema_demo

model:
  fields:
    - name: id
      type: integer
      required: true
    - name: value
      type: string

server:
  type: local
  path: "."
  schema_policy:
    # If an unknown field arrives, quarantine the row
    unknown_fields: "quarantine"
""",
    "01_data_quality_trust_demo/schema_policy.yaml",
)

# Upstream sends data with an undocumented 'hacked_payload' column
drifty_df = pl.DataFrame(
    {"id": [1, 2, 3], "value": ["a", "b", "c"], "hacked_payload": ["secret_1", "secret_2", "secret_3"]}
)

proc = ll.DataProcessor(schema_contract, engine="polars")
good, bad = proc.run(drifty_df)

print(f"\nGood Rows: {good.shape[0]} (Allowed)")
print(f"Quarantined Rows: {bad.shape[0]} (Due to unknown field)")
if bad.shape[0] > 0:
    print(f"Quarantine Reason: {bad['_lakelogic_errors'][0]}")

display(bad)

## What You Just Saw

- **100% reconciliation** — mathematical proof, not trust
- **Pydantic validation** — contract errors caught at load time
- **SQL-first rules** — 1 line replaces 20 lines of Python
- **SLO monitoring** — freshness and row count SLAs enforced automatically
- **Schema Drift Protection** — Schema Strictness & Unknown Field Quarantine

---
## Go Deeper — Explore by Capability

Each notebook below maps to a pillar of LakeLogic's [Technical Capabilities](https://lakelogic.github.io/LakeLogic/#technical-capabilities):

| # | Notebook | What You'll See |
|---|---|---|
| 🛡️ | **[Data Quality & Trust](01_data_quality_trust.ipynb)** | Reconciliation proofs, Pydantic validation, SQL-first rules, SLO monitoring |
| 📜 | **[Compliance & Governance](02_compliance_governance.ipynb)** | GDPR erasure in 2 lines, automatic lineage, cost intelligence |
| ⚡ | **[Engine & Scale](03_engine_scale.ipynb)** | Same contract on Polars & DuckDB, incremental processing, dry run |
| 🔧 | **[Developer Experience](04_developer_experience.ipynb)** | Structured diagnostics, DDL generation, surgical resets, multi-channel alerts |
| 🧬 | **[Data Generation & AI](05_data_generation_ai.ipynb)** | Synthetic data, referential integrity, edge case injection, contract inference |
| 🔌 | **[Integrations](06_integrations.ipynb)** | dbt adapter, dlt sources, contract-driven quality gates on arrival |

> **Each notebook is self-contained** — pick the capability that matters most to you and run it independently.